In [1]:
# Cell 1: Imports
import numpy as np               # NumPy: For numerical operations, especially arrays and matrices.
import pandas as pd              # Pandas: For data manipulation and analysis, particularly DataFrames.
import random                    # random: For generating pseudo-random numbers and random choices.
import matplotlib.pyplot as plt  # Matplotlib.pyplot: For creating static, animated, and interactive visualizations.
from collections import deque    # collections.deque: A double-ended queue for efficient appends and pops from both ends.
import sys                       # sys: For accessing system-specific parameters and functions.
import time                      # time: For time-related functions, like getting current time or pausing execution.


In [2]:
# Cell 2: HybridPowerFlowOptimizer Class (Modified)

class HybridPowerFlowOptimizer:
    """
    Optimizes power flow using a hybrid metaheuristic approach, incorporating
    local search, adaptive algorithm selection, stagnation handling, and
    adaptive parameters. Allows zero-cost load shedding.
    """

    def __init__(self, A_matrix, line_limits, gen_costs, gen_limits_min, gen_limits_max, initial_B, memory_size=100):
        """
        Initializes the HybridPowerFlowOptimizer.
        """
        self.A = A_matrix
        self.line_limits = np.array(line_limits, dtype=np.float64)
        self.num_lines = A_matrix.shape[0]
        self.num_buses = A_matrix.shape[1]
        self.tolerance = 1e-6

        self.initial_B = initial_B.copy().flatten()

        self.gen_indices = np.where(self.initial_B > self.tolerance)[0]
        self.load_indices = np.where(self.initial_B <= self.tolerance)[0]

        if len(self.gen_indices) == 0:
            raise ValueError("No generator buses identified (initial B > 0). Cannot proceed.")

        print(f"Identified {len(self.gen_indices)} generator buses (indices: {self.gen_indices})")
        print(f"Identified {len(self.load_indices)} load buses (indices: {self.load_indices}) - Load shedding enabled (Zero Cost).")

        flat_gc = np.array(gen_costs).flatten()
        flat_gmin = np.array(gen_limits_min).flatten()
        flat_gmax = np.array(gen_limits_max).flatten()
        if len(flat_gc) != self.num_buses or len(flat_gmin) != self.num_buses or len(flat_gmax) != self.num_buses:
            raise ValueError(f"Generator cost/limit array length mismatch (Expected {self.num_buses})")
        self.gen_costs_only = flat_gc[self.gen_indices]
        self.gen_limits_min_only = flat_gmin[self.gen_indices]
        self.gen_limits_max_only = flat_gmax[self.gen_indices]
        if np.any(self.gen_limits_max_only < self.gen_limits_min_only):
            raise ValueError("Generator Max limit cannot be less than Min limit.")

        self.initial_B_load_fixed = self.initial_B[self.load_indices]
        self.load_limits_min_only = self.initial_B_load_fixed
        self.load_limits_max_only = np.zeros_like(self.load_limits_min_only)

        needs_adjust = False

        # ==================== MODIFICATION START ====================
        #
        # The initial clamping of generator values has been disabled as per your request.
        # The optimizer will now start with the raw initial values, and the fitness function
        # will penalize any violations of generator limits from the very beginning.

        # # Clamp initial generator values to their min/max limits if they are outside.
        # initial_gen_values = self.initial_B[self.gen_indices]
        # if np.any(initial_gen_values < self.gen_limits_min_only - self.tolerance) or \
        #    np.any(initial_gen_values > self.gen_limits_max_only + self.tolerance):
        #     print("Warning: Initial generator B outside limits. Clamping...")
        #     self.initial_B[self.gen_indices] = np.clip(initial_gen_values, self.gen_limits_min_only, self.gen_limits_max_only)
        #     needs_adjust = True
        #
        # ===================== MODIFICATION END =====================

        required_total_injection = 0.0
        current_total_injection = np.sum(self.initial_B)
        difference_init = required_total_injection - current_total_injection
        num_gens = len(self.gen_indices)

        if abs(difference_init) > self.tolerance * self.num_buses:
            print(f"Warning: Initial injections sum to {current_total_injection:.4f} (≠ 0). Adjusting generators to balance...")
            if num_gens > 0:
                diff_per_gen_init = difference_init / num_gens
                adjusted_gens = self.initial_B[self.gen_indices] + diff_per_gen_init
                clipped_adjusted_gens = np.clip(adjusted_gens, self.gen_limits_min_only, self.gen_limits_max_only)
                actual_adjustment_applied = np.sum(clipped_adjusted_gens) - np.sum(self.initial_B[self.gen_indices])
                self.initial_B[self.gen_indices] = clipped_adjusted_gens
                remaining_diff = difference_init - actual_adjustment_applied
                if abs(remaining_diff) > self.tolerance * self.num_buses:
                    print(f"Warning: Could not fully balance initial state due to gen limits. Remaining imbalance: {remaining_diff:.4f}")
                needs_adjust = True
            else:
                print("ERROR: Cannot balance initial state - no generators identified.")
                needs_adjust = True

        if needs_adjust:
            print("-> Adjusted initial B state used for optimization:", np.round(self.initial_B, 4))

        self.memory = deque(maxlen=memory_size)
        self.stagnation_threshold = 50
        self.diversification_fraction = 0.2

    # --- NO OTHER CHANGES ARE NEEDED IN THE REST OF THE CLASS ---
    # (The rest of the methods: _apply_constraints, _calculate_fitness, etc., remain the same)
    
    def _apply_constraints(self, solution_vector):
        """
        Applies generator/load limits and enforces power balance on a solution vector.
        """
        sol = solution_vector.copy().flatten()
        num_gens = len(self.gen_indices)
        num_loads = len(self.load_indices)

        if num_loads > 0:
            try:
                sol[self.load_indices] = np.clip(sol[self.load_indices], self.load_limits_min_only, self.load_limits_max_only)
            except IndexError:
                print("Warning (_apply_constraints): Load index error.")
                pass

        if num_gens > 0:
            try:
                sol[self.gen_indices] = np.clip(sol[self.gen_indices], self.gen_limits_min_only, self.gen_limits_max_only)
            except IndexError:
                print("Warning (_apply_constraints): Gen index error.")
                pass

        if num_gens > 0:
            current_load_sum = np.sum(sol[self.load_indices]) if num_loads > 0 else 0.0
            required_gen_sum = -current_load_sum
            current_gen_sum = np.sum(sol[self.gen_indices])
            difference = required_gen_sum - current_gen_sum

            if abs(difference) > self.tolerance * self.num_buses:
                adjustment_per_gen = difference / num_gens
                adjusted_gens = sol[self.gen_indices] + adjustment_per_gen
                clipped_adjusted_gens = np.clip(adjusted_gens, self.gen_limits_min_only, self.gen_limits_max_only)
                actual_adjustment_applied = np.sum(clipped_adjusted_gens) - current_gen_sum
                remaining_diff = difference - actual_adjustment_applied
                sol[self.gen_indices] = clipped_adjusted_gens
                if abs(remaining_diff) > self.tolerance * self.num_buses * 10:
                    pass

        return sol.reshape(-1, 1)

    def _calculate_fitness(self, solution):
        """
        Calculates the fitness of a solution. Lower fitness is better.
        """
        sol_flat = solution.flatten()
        if not np.all(np.isfinite(sol_flat)): return np.inf

        try:
            C = np.dot(self.A, sol_flat)
            if not np.all(np.isfinite(C)): return np.inf
            flows = C.flatten()
        except ValueError:
            return np.inf

        penalty_multiplier = 1e10

        line_violations = np.maximum(0, np.abs(flows) - (self.line_limits + self.tolerance))
        line_violation_penalty = penalty_multiplier * np.sum(line_violations**2)

        gen_limit_penalty = 0.0
        if len(self.gen_indices) > 0:
            gen_values = sol_flat[self.gen_indices]
            violations_lower_g = np.maximum(0, self.gen_limits_min_only - gen_values + self.tolerance)
            violations_upper_g = np.maximum(0, gen_values - self.gen_limits_max_only - self.tolerance)
            gen_limit_penalty = penalty_multiplier * (np.sum(violations_lower_g**2) + np.sum(violations_upper_g**2))

        load_limit_penalty = 0.0
        if len(self.load_indices) > 0:
            load_values = sol_flat[self.load_indices]
            violations_lower_l = np.maximum(0, self.load_limits_min_only - load_values + self.tolerance)
            violations_upper_l = np.maximum(0, load_values - self.load_limits_max_only - self.tolerance)
            load_limit_penalty = penalty_multiplier * (np.sum(violations_lower_l**2) + np.sum(violations_upper_l**2))

        balance_violation = abs(np.sum(sol_flat))
        balance_penalty = penalty_multiplier * (balance_violation**2) if balance_violation > self.tolerance * self.num_buses else 0.0

        generator_deviation = self._get_generator_deviation(solution)
        deviation_penalty_component = 1e5 * generator_deviation

        gen_rescheduling_cost = self._get_rescheduling_cost(solution)
        gen_cost_weight = 0.1

        fitness = (line_violation_penalty
                   + gen_limit_penalty
                   + load_limit_penalty
                   + balance_penalty
                   + deviation_penalty_component
                   + gen_cost_weight * gen_rescheduling_cost
                   )

        return fitness if np.isfinite(fitness) else np.inf

    def _is_feasible(self, solution, verbose=False):
        """
        Checks if a solution meets all hard constraints within tolerance.
        """
        if solution is None:
            if verbose: print("DEBUG (_is_feasible): Input solution is None.")
            return False
        sol_flat = solution.flatten()
        if not np.all(np.isfinite(sol_flat)):
            if verbose: print("DEBUG (_is_feasible): Solution contains non-finite values.")
            return False

        line_ok = False
        flows = np.array([])
        try:
            C = np.dot(self.A, sol_flat)
            if not np.all(np.isfinite(C)): raise ValueError("Flows NaN/Inf")
            flows = C.flatten()
            line_ok = np.all(np.abs(flows) <= self.line_limits + self.tolerance)
        except Exception as e:
            line_ok = False
            if verbose: print(f"DEBUG: Line check error: {e}")

        gen_ok = False
        gen_values = sol_flat[self.gen_indices] if len(self.gen_indices) > 0 else np.array([])
        if len(gen_values) == len(self.gen_limits_min_only):
            gen_ok = np.all(gen_values >= self.gen_limits_min_only - self.tolerance) and \
                     np.all(gen_values <= self.gen_limits_max_only + self.tolerance)
        elif len(self.gen_indices) == 0:
            gen_ok = True

        load_ok = False
        load_values = sol_flat[self.load_indices] if len(self.load_indices) > 0 else np.array([])
        if len(load_values) == len(self.load_limits_min_only):
            load_ok = np.all(load_values <= self.load_limits_max_only + self.tolerance) and \
                      np.all(load_values >= self.load_limits_min_only - self.tolerance)
        elif len(self.load_indices) == 0:
            load_ok = True

        bal_ok = abs(np.sum(sol_flat)) < self.tolerance * self.num_buses

        feasible = line_ok and gen_ok and load_ok and bal_ok

        if verbose or not feasible:
            print(f"--- Feasibility Check {'FAILED' if not feasible else 'PASSED'} ---")
            if not line_ok:
                failing_lines = np.where(np.abs(flows) > self.line_limits + self.tolerance)[0]
                print(f"  Line constraints failed. Failing lines: {failing_lines+1}")
                for idx in failing_lines[:5]:
                    if idx < len(flows) and idx < len(self.line_limits):
                          print(f"    Line {idx+1}: Flow={abs(flows[idx]):.4f}, Limit={self.line_limits[idx]:.4f}")
                    else: print(f"    Line {idx+1}: Index out of bounds for details.")
            else: print("  Line constraints met.")

            if not gen_ok:
                if len(gen_values) == len(self.gen_limits_min_only):
                    failing_min_g = np.where(gen_values < self.gen_limits_min_only - self.tolerance)[0]
                    failing_max_g = np.where(gen_values > self.gen_limits_max_only + self.tolerance)[0]
                    if len(failing_min_g) > 0: print(f"  Gen Bus {self.gen_indices[failing_min_g[0]]+1} violated MIN Limit ({gen_values[failing_min_g[0]]:.3f} < {self.gen_limits_min_only[failing_min_g[0]]:.3f})")
                    elif len(failing_max_g) > 0: print(f"  Gen Bus {self.gen_indices[failing_max_g[0]]+1} violated MAX Limit ({gen_values[failing_max_g[0]]:.3f} > {self.gen_limits_max_only[failing_max_g[0]]:.3f})")
                    else: print("  Generator limits failed (Unknown reason).")
                else: print(f"  Generator limits failed (Dimension mismatch: Expected {len(self.gen_limits_min_only)}, Got {len(gen_values)}).")
            else: print("  Generator limits met.")

            if not load_ok:
                if len(load_values) == len(self.load_limits_min_only):
                    failing_min_l = np.where(load_values < self.load_limits_min_only - self.tolerance)[0]
                    failing_max_l = np.where(load_values > self.load_limits_max_only + self.tolerance)[0]
                    if len(failing_min_l)>0: print(f"  Load Bus {self.load_indices[failing_min_l[0]]+1} violated MIN Limit ({load_values[failing_min_l[0]]:.3f} < {self.load_limits_min_only[failing_min_l[0]]:.3f})")
                    elif len(failing_max_l)>0: print(f"  Load Bus {self.load_indices[failing_max_l[0]]+1} violated MAX Limit ({load_values[failing_max_l[0]]:.3f} > {self.load_limits_max_only[failing_max_l[0]]:.3f})")
                    else: print("  Load limits failed (Unknown reason).")
                else: print(f"  Load limits failed (Dimension mismatch: Expected {len(self.load_limits_min_only)}, Got {len(load_values)}).")
            else: print("  Load limits met.")

            if not bal_ok: print(f"  Power balance check failed, sum={np.sum(sol_flat):.8f}")
            else: print("  Power balance met.")
            print("-" * 40)
        return feasible

    def _get_generator_deviation(self, solution):
        """
        Calculates the sum of absolute changes in generator outputs.
        """
        sol_flat = solution.flatten()
        if len(self.gen_indices) == 0: return 0.0
        try:
            if np.max(self.gen_indices) >= len(sol_flat) or np.max(self.gen_indices) >= len(self.initial_B):
                print("Warning (_get_generator_deviation): Index out of bounds.")
                return np.inf
            return np.sum(np.abs(sol_flat[self.gen_indices] - self.initial_B[self.gen_indices]))
        except IndexError:
            print("Warning (_get_generator_deviation): Index error during calculation.")
            return np.inf

    def _get_rescheduling_cost(self, solution):
        """
        Calculates the cost associated with changing generator outputs.
        """
        sol_flat = solution.flatten()
        if len(self.gen_indices) == 0: return 0.0
        try:
            if len(self.gen_costs_only) != len(self.gen_indices) or \
               np.max(self.gen_indices) >= len(sol_flat) or \
               np.max(self.gen_indices) >= len(self.initial_B):
                print("Warning (_get_rescheduling_cost): Index or dimension mismatch.")
                return np.inf
            deviation = np.abs(sol_flat[self.gen_indices] - self.initial_B[self.gen_indices])
            cost = np.sum(self.gen_costs_only * deviation)
            return cost
        except IndexError:
            print("Warning (_get_rescheduling_cost): Index error during calculation.")
            return np.inf

    def _generate_random_solution(self):
        """
        Generates a new random solution, perturbed around the initial state.
        """
        rand_sol = self.initial_B.copy()
        n_gens = len(self.gen_indices)
        n_loads = len(self.load_indices)

        if n_gens > 0:
            gen_range = np.maximum(self.tolerance, self.gen_limits_max_only - self.gen_limits_min_only)
            perturbation_factor = 0.2
            perturbations = (np.random.rand(n_gens) - 0.5) * gen_range * perturbation_factor
            rand_sol[self.gen_indices] = np.clip(self.initial_B[self.gen_indices] + perturbations,
                                                  self.gen_limits_min_only, self.gen_limits_max_only)

        if n_loads > 0:
            shedding_factor = 0.1
            potential_increase = np.random.rand(n_loads) * np.abs(self.initial_B[self.load_indices]) * shedding_factor
            rand_sol[self.load_indices] = np.clip(self.initial_B[self.load_indices] + potential_increase,
                                                   self.load_limits_min_only, self.load_limits_max_only)

        return self._apply_constraints(rand_sol)

    def _apply_local_search(self, solution, iteration, max_iterations):
        """
        Applies a simple local search heuristic to refine a solution.
        """
        current_solution = solution.copy()
        current_fitness = self._calculate_fitness(current_solution)
        if not np.isfinite(current_fitness): return solution

        progress_ratio = iteration / max_iterations
        step_scale_factor = 0.1 * (1.0 - progress_ratio) + 0.01 * progress_ratio
        num_attempts = min(5, int(np.sqrt(self.num_buses)))

        for _ in range(num_attempts):
            idx_to_perturb = random.randrange(self.num_buses)
            perturbed_solution = current_solution.copy()
            original_value = perturbed_solution[idx_to_perturb, 0]

            perturb_range = 1.0
            if idx_to_perturb in self.gen_indices:
                gen_idx_local = np.where(self.gen_indices == idx_to_perturb)[0][0]
                gen_op_range = self.gen_limits_max_only[gen_idx_local] - self.gen_limits_min_only[gen_idx_local]
                perturb_range = max(self.tolerance, gen_op_range) * step_scale_factor
            elif idx_to_perturb in self.load_indices:
                load_idx_local = np.where(self.load_indices == idx_to_perturb)[0][0]
                load_op_range = self.load_limits_max_only[load_idx_local] - self.load_limits_min_only[load_idx_local]
                perturb_range = max(self.tolerance, load_op_range) * step_scale_factor

            perturbation = (random.random() - 0.5) * 2 * perturb_range
            perturbed_solution[idx_to_perturb, 0] += perturbation

            refined_solution = self._apply_constraints(perturbed_solution)
            refined_fitness = self._calculate_fitness(refined_solution)

            if np.isfinite(refined_fitness) and refined_fitness < current_fitness:
                current_solution = refined_solution
                current_fitness = refined_fitness
        return current_solution

    def optimize(self, B_unused, iterations=200, population_size=30):
        """
        Main optimization loop.
        """
        print(f"Starting Optimization: Pop={population_size}, Iterations={iterations}")

        initial_state_constrained = self._apply_constraints(self.initial_B)
        initial_population = [self._generate_random_solution() for _ in range(population_size)]
        initial_population[0] = initial_state_constrained
        fitness_values = [self._calculate_fitness(sol) for sol in initial_population]
        best_idx = np.argmin(fitness_values)
        best_solution = initial_population[best_idx].copy()
        best_fitness = fitness_values[best_idx]
        if not np.isfinite(best_fitness):
            print("ERROR: Initial best fitness is non-finite. Cannot proceed with this population.")
            finite_indices = np.where(np.isfinite(fitness_values))[0]
            if len(finite_indices) > 0:
                best_idx = finite_indices[np.argmin(np.array(fitness_values)[finite_indices])]
                best_solution = initial_population[best_idx].copy()
                best_fitness = fitness_values[best_idx]
                print(f"Warning: Using first finite solution found as initial best (Fitness: {best_fitness:.4e})")
            else:
                print("FATAL: No finite fitness found in initial population. Returning initial B.")
                return self.initial_B.reshape(-1, 1), [], np.dot(self.A, self.initial_B.reshape(-1,1))

        print(f"Initial Best Fitness: {best_fitness:.4e}")
        fitness_history = [best_fitness]
        
        algorithm_names = ['OOA', 'KHA', 'SHO']
        algo_success_count = {name: 0 for name in algorithm_names}
        algo_attempts_count = {name: 0 for name in algorithm_names}
        algo_probabilities = {name: 1.0/len(algorithm_names) for name in algorithm_names}
        adaptation_rate = 0.05
        stagnation_counter = 0
        last_best_fitness = best_fitness

        for iteration in range(iterations):
            new_population = []
            current_fitness_values = []

            total_attempts = sum(algo_attempts_count.values())
            if total_attempts > 0:
                success_rates = {name: algo_success_count[name] / algo_attempts_count[name]
                                 if algo_attempts_count[name] > 0 else 0
                                 for name in algorithm_names}
                total_rate = sum(success_rates.values())
                if total_rate > 1e-6:
                    target_probabilities = {name: rate / total_rate for name, rate in success_rates.items()}
                    for name in algorithm_names:
                        algo_probabilities[name] = (1 - adaptation_rate) * algo_probabilities[name] + \
                                                   adaptation_rate * target_probabilities[name]
                    prob_sum = sum(algo_probabilities.values())
                    if prob_sum > 1e-6:
                        algo_probabilities = {name: p / prob_sum for name, p in algo_probabilities.items()}
                    else:
                        algo_probabilities = {name: 1.0/len(algorithm_names) for name in algorithm_names}
            current_weights = [algo_probabilities[name] for name in algorithm_names]
            if not (len(current_weights) == len(algorithm_names) and abs(sum(current_weights) - 1.0) < 1e-5 and all(w >= 0 for w in current_weights)):
                current_weights = [1.0/len(algorithm_names)] * len(algorithm_names)

            population_fitness_improved_count = 0
            for i in range(population_size):
                try:
                    algorithm = random.choices(algorithm_names, weights=current_weights, k=1)[0]
                except ValueError:
                    print("Warning: Issue with adaptive weights, using equal probability.")
                    algorithm = random.choice(algorithm_names)

                algo_attempts_count[algorithm] += 1
                current_solution = initial_population[i]
                initial_fitness_of_parent = fitness_values[i]

                if algorithm == 'OOA':
                    candidate_solution = self._apply_orcas_optimization(initial_population, i, best_solution, iteration, iterations)
                elif algorithm == 'KHA':
                    candidate_solution = self._apply_krill_herd(initial_population, i, best_solution, iteration, iterations)
                else: # SHO
                    candidate_solution = self._apply_spotted_hyena(initial_population, i, best_solution, iteration, iterations)

                refined_candidate = self._apply_local_search(candidate_solution, iteration, iterations)
                new_solution = self._apply_constraints(refined_candidate)
                new_population.append(new_solution)
                new_fitness = self._calculate_fitness(new_solution)
                current_fitness_values.append(new_fitness)

                if np.isfinite(new_fitness) and new_fitness < best_fitness:
                    best_fitness = new_fitness
                    best_solution = new_solution.copy()
                    algo_success_count[algorithm] += 1
                    population_fitness_improved_count +=1

            initial_population = new_population
            fitness_values = current_fitness_values

            if np.isfinite(best_fitness):
                fitness_history.append(best_fitness)
            
            if abs(best_fitness - last_best_fitness) < self.tolerance * max(1.0, abs(last_best_fitness)):
                stagnation_counter += 1
            else:
                stagnation_counter = 0
                last_best_fitness = best_fitness

            if stagnation_counter >= self.stagnation_threshold:
                print(f"\nStagnation detected at iteration {iteration}. Diversifying population...")
                num_to_replace = int(population_size * self.diversification_fraction)
                worst_indices = np.argsort(fitness_values)[-num_to_replace:]
                for idx in worst_indices:
                    initial_population[idx] = self._generate_random_solution()
                    fitness_values[idx] = self._calculate_fitness(initial_population[idx])

                current_best_idx_after_diversify = np.argmin(fitness_values)
                if fitness_values[current_best_idx_after_diversify] < best_fitness:
                    best_solution = initial_population[current_best_idx_after_diversify].copy()
                    best_fitness = fitness_values[current_best_idx_after_diversify]
                last_best_fitness = best_fitness
                stagnation_counter = 0

            if iteration % 100 == 0 or iteration == iterations - 1:
                deviation = self._get_generator_deviation(best_solution)
                cost = self._get_rescheduling_cost(best_solution)
                print(f"Iter {iteration}/{iterations}: BestFit={best_fitness:.4e}, Deviation={deviation:.3f}, GenCost={cost:.2f} (Stagnation: {stagnation_counter}/{self.stagnation_threshold})")

        best_solution = self._apply_constraints(best_solution)

        total_attempts_final = sum(algo_attempts_count.values())
        if total_attempts_final > 0:
            print("\nAlgorithm Contributions (Attempts):", {k: f"{v/total_attempts_final*100:.1f}% ({v})" for k, v in algo_attempts_count.items()})
        total_success_final = sum(algo_success_count.values())
        if total_attempts_final > 0:
             print("Algorithm Contributions (Global Bests Found):", {k: f"{algo_success_count[k]} / {algo_attempts_count[k]}" for k in algorithm_names})

        final_deviation = self._get_generator_deviation(best_solution)
        final_gen_cost = self._get_rescheduling_cost(best_solution)
        print(f"\nFinal Sum of Absolute Changes (Generators Only): {final_deviation:.4f}")
        print(f"Final Generator Rescheduling Cost: {final_gen_cost:.2f} $/hr")

        final_C = np.dot(self.A, best_solution)
        return best_solution, fitness_history, final_C

    def _check_memory(self, B_unused_param_for_consistency_with_a_potential_interface):
        """
        Checks memory for a previously stored feasible solution.
        """
        if not self.memory: return None
        for _, stored_B in reversed(self.memory):
            if self._is_feasible(stored_B):
                print("Using feasible solution from memory.")
                return stored_B
        return None

    def _store_in_memory(self, initial_B_state, optimized_B_state):
        """
        Stores a feasible solution pair (initial and optimized) in memory.
        """
        self.memory.append((initial_B_state.copy(), optimized_B_state.copy()))
        print(f"Feasible solution stored. Memory size: {len(self.memory)}")

    def _apply_orcas_optimization(self, population, current_index, best_solution_global, iteration, max_iterations):
        """
        Applies a step of the Orca Optimization Algorithm (OOA).
        """
        solution = population[current_index].copy()
        best_sol_flat = best_solution_global.flatten()
        sol_flat = solution.flatten()

        a = 2 * (1 - (iteration / max_iterations)**2)
        r1, r2 = random.random(), random.random()

        if r1 < 0.5:
            step = a * r2 * (best_sol_flat - sol_flat)
            new_solution_flat = sol_flat + step
        else:
            other_indices = [j for j in range(len(population)) if j != current_index]
            random_solution_idx = random.choice(other_indices) if other_indices else current_index
            random_solution = population[random_solution_idx].flatten()
            A_param = 2 * a * r1 - a
            C_param = 2 * r2
            D_param = np.abs(C_param * random_solution - sol_flat)
            new_solution_flat = random_solution - A_param * D_param

        new_solution_flat = np.nan_to_num(new_solution_flat, nan=np.mean(sol_flat), posinf=np.max(sol_flat)*2, neginf=np.min(sol_flat)*2)
        return new_solution_flat.reshape(-1, 1)

    def _apply_krill_herd(self, population, current_index, best_solution_global, iteration, max_iterations):
        """
        Applies a step of the Krill Herd Algorithm (KHA).
        """
        solution = population[current_index].copy()
        best_sol_flat = best_solution_global.flatten()
        sol_flat = solution.flatten()

        Dmax = 0.005 * (1 - iteration / max_iterations)
        Vf = 0.02
        Nmax = 0.01
        Dt = 1.0

        N_induced = Nmax * (best_sol_flat - sol_flat)
        F_foraging = Vf * (best_sol_flat - sol_flat)
        diffusion_vector = np.random.uniform(-1, 1, sol_flat.shape)
        D_physical = Dmax * diffusion_vector
        new_solution_flat = sol_flat + Dt * (N_induced + F_foraging + D_physical)

        new_solution_flat = np.nan_to_num(new_solution_flat, nan=np.mean(sol_flat), posinf=np.max(sol_flat)*2, neginf=np.min(sol_flat)*2)
        return new_solution_flat.reshape(-1, 1)

    def _apply_spotted_hyena(self, population, current_index, best_solution_global, iteration, max_iterations):
        """
        Applies a step of the Spotted Hyena Optimizer (SHO).
        """
        solution = population[current_index].copy()
        best_sol_flat = best_solution_global.flatten()
        sol_flat = solution.flatten()

        h = 5 - iteration * (5 / max_iterations)
        B_param = 2 * random.random()
        E_param = 2 * h * random.random() - h
        D_best = np.abs(B_param * best_sol_flat - sol_flat)
        X1 = best_sol_flat - E_param * D_best

        if abs(E_param) >= 1:
            other_indices = [j for j in range(len(population)) if j != current_index]
            random_hyena_idx = random.choice(other_indices) if other_indices else current_index
            random_hyena = population[random_hyena_idx].flatten()
            D_hyena = np.abs(B_param * random_hyena - sol_flat)
            new_solution_flat = random_hyena - E_param * D_hyena
        else:
            new_solution_flat = X1

        new_solution_flat = np.nan_to_num(new_solution_flat, nan=np.mean(sol_flat), posinf=np.max(sol_flat)*2, neginf=np.min(sol_flat)*2)
        return new_solution_flat.reshape(-1, 1)

In [3]:
# Cell 3: Wrapper Function
def optimize_power_flow_free_loadshed(A, B, line_limits, gen_costs, gen_limits_min, gen_limits_max,
                                      iterations=5000, population_size=100):
    """
    Wrapper function to initialize and run the HybridPowerFlowOptimizer.
    The primary objective is to satisfy all operational constraints (line limits,
    generator limits, power balance) while minimizing the deviation of generator
    outputs from their initial setpoints. A secondary objective is to minimize
    the rescheduling cost associated with these deviations.
    This version allows for load shedding at zero cost, meaning loads can be
    reduced from their initial values down to zero if necessary to meet constraints
    or achieve a better overall solution according to the primary objectives.

    Args:
        A (np.ndarray): System matrix (PTDF or similar), shape (n_lines, n_buses).
        B (np.ndarray): Initial bus injections (n_buses x 1). Positive for generation, negative for load.
        line_limits (np.ndarray): Absolute limits for line flows (n_lines).
        gen_costs (np.ndarray): Cost coefficients for each bus (n_buses). Only used for generator buses.
                                Represents cost per unit of deviation.
        gen_limits_min (np.ndarray): Min generation limit for each bus (n_buses). Applied to generator buses.
        gen_limits_max (np.ndarray): Max generation limit for each bus (n_buses). Applied to generator buses.
        iterations (int): Number of optimization iterations.
        population_size (int): Number of solutions in the population for metaheuristic algorithms.

    Returns:
        tuple: (B_optimized, fitness_history, C_optimized, C_unoptimized, final_feasible, opt_details)
               - B_optimized (np.ndarray): Optimized bus injections.
               - fitness_history (list): List of best fitness values per iteration.
               - C_optimized (np.ndarray): Line flows corresponding to B_optimized.
               - C_unoptimized (np.ndarray): Line flows corresponding to the (potentially adjusted) initial B.
               - final_feasible (bool): Boolean indicating if the final optimized solution is feasible.
               - opt_details (dict): Dictionary containing 'gen_indices' and 'load_indices' identified by the optimizer.
    """
    print("--- Initializing Optimizer (Enhanced Hybrid - Load Shed Allowed - Zero Cost) ---")
    try:
        # Ensure the optimizer class is defined (e.g., Cell 2 has been run)
        if 'HybridPowerFlowOptimizer' not in globals():
            raise NameError("Optimizer class 'HybridPowerFlowOptimizer' is not defined.")

        # Instantiate the optimizer
        optimizer = HybridPowerFlowOptimizer(A, line_limits, gen_costs, gen_limits_min, gen_limits_max, B)

    except (ValueError, NameError) as e: # Catch errors during optimizer initialization
        print(f"ERROR initializing optimizer: {e}")
        # Fallback: return initial B and empty/NaN results if optimizer fails to init
        try:
            C_unoptimized_calc = np.dot(A, B) # Try to calculate initial flows
        except Exception: # If even this fails (e.g. A, B shape mismatch)
            C_unoptimized_calc = np.full((A.shape[0] if hasattr(A, 'shape') else 1, 1), np.nan)
        return B, [], C_unoptimized_calc, C_unoptimized_calc, False, {}

    # Get the initial B state as processed by the optimizer (e.g., after clamping, balancing)
    initial_B_from_opt = optimizer.initial_B.reshape(-1, 1)
    try:
        # Calculate line flows for this (optimizer's) initial state
        C_unoptimized = np.dot(A, initial_B_from_opt)
    except Exception as e:
        print(f"Warning: Could not calculate initial flows with optimizer's initial B: {e}")
        C_unoptimized = np.full((A.shape[0], 1), np.nan) # Fallback for C_unoptimized

    print("\n--- Checking Initial State Feasibility (Using Optimizer's Initial State) ---")
    # Check if the optimizer's starting point is feasible
    initial_feasible = optimizer._is_feasible(initial_B_from_opt, verbose=True)
    print(f"Optimizer's initial state feasible: {initial_feasible}")
    if not initial_feasible:
        print("WARNING: Optimizer starting from an infeasible state (may take longer or fail to find a feasible solution).")

    print("\n--- Starting Optimization (Enhanced Hybrid - Load Shed Allowed - Zero Cost) ---")
    # Run the main optimization process
    # The first argument to optimizer.optimize (B_unused) is not used by this implementation's optimize method.
    B_opt, fit_hist, C_opt = optimizer.optimize(None, iterations=iterations, population_size=population_size)

    print("\n--- Checking Final Solution Feasibility ---")
    # Check if the final optimized solution is feasible
    final_feasible = optimizer._is_feasible(B_opt, verbose=True)
    print(f"\nFinal feasibility: {final_feasible}")

    # Store details like generator and load indices identified by the optimizer
    details = {"gen_indices": optimizer.gen_indices, "load_indices": optimizer.load_indices}

    return B_opt, fit_hist, C_opt, C_unoptimized, final_feasible, details


In [4]:
# Cell 4: Visualization Function (Modified)

def visualize_results(A, B, B_optimized, C_optimized, C_unoptimized, line_limits,
                      gen_costs, gen_limits_min, gen_limits_max,
                      gen_indices, load_indices, fitness_history):
    """
    Visualizes the optimization results, including line flows, bus injections,
    fitness convergence, and changes in generation/load.
    Includes checks for valid data before attempting to plot.

    Args:
        A (np.ndarray): System matrix.
        B (np.ndarray): Initial bus injections.
        B_optimized (np.ndarray): Optimized bus injections.
        C_optimized (np.ndarray): Optimized line flows.
        C_unoptimized (np.ndarray): Unoptimized (initial) line flows.
        line_limits (np.ndarray): Line flow limits.
        gen_costs (np.ndarray): Generator cost coefficients (per bus).
        gen_limits_min (np.ndarray): Generator minimum limits (per bus).
        gen_limits_max (np.ndarray): Generator maximum limits (per bus).
        gen_indices (np.ndarray): Indices of generator buses.
        load_indices (np.ndarray): Indices of load buses.
        fitness_history (list): History of best fitness values during optimization.
    """
    num_lines = A.shape[0]
    num_buses = A.shape[1]
    tolerance = 1e-6 # Tolerance for checking violations in plots
    print("\n--- Generating Plots (Enhanced Hybrid - Load Shed Allowed - Zero Cost) ---")

    # MODIFIED Plot 1: Line Flows Comparison as a Bar Graph
    try:
        plt.figure(figsize=(14, 7))
        idx = np.arange(1, num_lines + 1)  # X-axis for line indices (1-based)
        bar_width = 0.4

        # Plot unoptimized flows, now colored by their violation status
        if isinstance(C_unoptimized, np.ndarray) and C_unoptimized.size == num_lines:
            unoptimized_flows_abs = np.abs(C_unoptimized.flatten())
            # Determine colors based on violations for UNOPTIMIZED flows
            colors_unoptimized = ['crimson' if flow > limit + tolerance else 'skyblue'
                                  for flow, limit in zip(unoptimized_flows_abs, line_limits)]
            plt.bar(idx - bar_width / 2, unoptimized_flows_abs, width=bar_width,
                    label='Unoptimized Flow (Absolute)', color=colors_unoptimized)
        else:
            print("Warning: Skipping unoptimized flows plot (invalid data for C_unoptimized).")

        # Plot optimized flows with a single color (no special violation color)
        if isinstance(C_optimized, np.ndarray) and C_optimized.size == num_lines:
            optimized_flows_abs = np.abs(C_optimized.flatten())
            plt.bar(idx + bar_width / 2, optimized_flows_abs, width=bar_width,
                    label='Optimized Flow (Absolute)', color='forestgreen')
        else:
            print("Warning: Skipping optimized flows plot (invalid data for C_optimized).")

        # Plot line limits as a dashed line
        plt.plot(idx, line_limits, 'k--', alpha=0.9, label='Line Limit')

        plt.xlabel('Line Index')
        plt.ylabel('Absolute Power Flow (MW or p.u.)')
        plt.title('Line Flow Comparison (Bar Graph)')

        # Create a new custom legend to explain the updated colors
        import matplotlib.patches as mpatches
        from matplotlib.lines import Line2D

        unopt_ok_patch = mpatches.Patch(color='skyblue', label='Unoptimized (Within Limit)')
        unopt_viol_patch = mpatches.Patch(color='crimson', label='Unoptimized (Violation)')
        opt_patch = mpatches.Patch(color='forestgreen', label='Optimized Flow')
        limit_line = Line2D([0], [0], color='k', lw=2, linestyle='--', label='Line Limit')

        plt.legend(handles=[unopt_ok_patch, unopt_viol_patch, opt_patch, limit_line])

        plt.grid(True, axis='y', linestyle=':')
        plt.xticks(idx[::max(1, num_lines // 20)])
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Plotting error (Line Flows): {e}")


    # Plot 2: Fitness Convergence
    try:
        if isinstance(fitness_history, (list, np.ndarray)) and len(fitness_history) > 1:
            plt.figure(figsize=(10, 5))
            finite_fitness = [f for f in fitness_history if np.isfinite(f)]
            if len(finite_fitness) > 1:
                plt.plot(finite_fitness, '.-', color='royalblue', label='Best Fitness', markersize=3, linewidth=1)
                plt.xlabel('Iteration')
                plt.ylabel('Fitness Value (Log Scale)')
                plt.title('Optimization Convergence')
                plt.yscale('log')
                plt.legend()
                plt.grid(True, linestyle=':')
                plt.tight_layout()
                plt.show()
            else: print("Warning: Not enough finite fitness values to plot convergence.")
        else: print("Warning: Skipping fitness plot (insufficient or invalid history data).")
    except Exception as e:
        print(f"Plotting error (Fitness History): {e}")

    # Plot 3: Bus Power Injections (Initial vs. Optimized)
    try:
        plt.figure(figsize=(14, 7))
        bus_idx_plot = np.arange(1, num_buses + 1)
        bar_width = 0.35
        colors_initial = ['darkblue' if i in gen_indices else 'skyblue' for i in range(num_buses)]
        colors_optimized = ['darkgreen' if i in gen_indices else 'lightgreen' for i in range(num_buses)]
        if isinstance(B, np.ndarray) and B.size == num_buses:
            plt.bar(bus_idx_plot - bar_width/2, B.flatten(), width=bar_width, label='Initial (Gen=dark blue, Load=light blue)', alpha=0.7, color=colors_initial)
        else: print("Warning: Skipping initial injections plot (invalid data for B).")
        if isinstance(B_optimized, np.ndarray) and B_optimized.size == num_buses:
            plt.bar(bus_idx_plot + bar_width/2, B_optimized.flatten(), width=bar_width, label='Optimized (Gen=dark green, Load=light green)', alpha=0.8, color=colors_optimized)
        else: print("Warning: Skipping optimized injections plot (invalid data for B_optimized).")
        if len(gen_indices) > 0:
            if isinstance(gen_limits_max, np.ndarray) and len(gen_limits_max) == num_buses and \
               isinstance(gen_limits_min, np.ndarray) and len(gen_limits_min) == num_buses:
                plt.scatter(bus_idx_plot[gen_indices], gen_limits_max[gen_indices], c='dimgrey', marker='_', s=150, label='Gen Max Limit', zorder=5)
                plt.scatter(bus_idx_plot[gen_indices], gen_limits_min[gen_indices], c='dimgrey', marker='_', s=150, label='Gen Min Limit', zorder=5)
            else: print("Warning: Skipping generator limit markers (invalid limit data arrays).")
        if len(load_indices) > 0:
            if isinstance(B, np.ndarray) and B.size == num_buses:
                plt.scatter(bus_idx_plot[load_indices], np.zeros(len(load_indices)), c='lightcoral', marker='_', s=150, label='Load Max Limit (0)', zorder=5)
                plt.scatter(bus_idx_plot[load_indices], B.flatten()[load_indices], c='lightcoral', marker='_', s=150, label='Load Min Limit (Initial)', zorder=5)
            else: print("Warning: Skipping load limit markers (invalid initial B data for load limits).")
        plt.xlabel('Bus Index')
        plt.ylabel('Power Injection (MW or p.u.)')
        plt.title('Bus Power Injections: Initial vs. Optimized')
        plt.xticks(bus_idx_plot[::max(1, num_buses//20)])
        plt.legend()
        plt.grid(True, axis='y', linestyle=':')
        plt.axhline(0, color='black', linewidth=0.5)
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Plotting error (Bus Injections): {e}")

    # Plot 4 & 5: Changes in Generator Output and Load Shedding
    try:
        if isinstance(B, np.ndarray) and B.size == num_buses and \
           isinstance(B_optimized, np.ndarray) and B_optimized.size == num_buses:
            changes = B_optimized.flatten() - B.flatten()
            bus_idx_plot_full = np.arange(1, num_buses + 1)
            if len(gen_indices) > 0:
                plt.figure(figsize=(10, 4))
                gen_changes = changes[gen_indices]
                gen_labels = bus_idx_plot_full[gen_indices]
                colors_gen = ['forestgreen' if x >= 0 else 'firebrick' for x in gen_changes]
                bar_indices_gen = np.arange(len(gen_indices))
                plt.bar(bar_indices_gen, gen_changes, color=colors_gen)
                plt.axhline(0, color='black', linestyle='-', linewidth=0.7)
                plt.xlabel('Generator Bus Index')
                plt.ylabel('Change in Injection (Optimized - Initial)')
                plt.title('Generator Output Changes')
                plt.xticks(bar_indices_gen, gen_labels)
                plt.grid(True, axis='y', linestyle=':')
                plt.tight_layout()
                plt.show()
            if len(load_indices) > 0:
                plt.figure(figsize=(10, 4))
                load_changes = changes[load_indices]
                load_shed_values = np.maximum(0, load_changes)
                load_labels = bus_idx_plot_full[load_indices]
                colors_load = ['darkorange' if x > tolerance else 'darkgrey' for x in load_shed_values]
                bar_indices_load = np.arange(len(load_indices))
                plt.bar(bar_indices_load, load_shed_values, color=colors_load)
                plt.axhline(0, color='black', linestyle='-', linewidth=0.7)
                plt.xlabel('Load Bus Index')
                plt.ylabel('Load Shed Amount (MW or p.u.)')
                plt.title('Load Shedding (Positive value means load was reduced)')
                plt.xticks(bar_indices_load, load_labels)
                plt.grid(True, axis='y', linestyle=':')
                plt.ylim(bottom= -0.05 * max(1, np.max(load_shed_values)) if np.any(load_shed_values > 0) else -0.1)
                plt.tight_layout()
                plt.show()
        else:
            print("Warning: Skipping change plots (invalid B or B_optimized data).")
    except Exception as e:
        print(f"Plotting error (Changes in Injections/Load Shed): {e}")

    # --- Text Summary of Results ---
    print("\n" + "="*30 + " RESULTS SUMMARY (Enhanced Hybrid) " + "="*30)
    np.set_printoptions(precision=4, suppress=True)
    try:
        final_dev_g = 0.0
        final_g_cost = 0.0
        total_ls = 0.0
        if len(gen_indices) > 0 and isinstance(B_optimized, np.ndarray) and B_optimized.size==num_buses \
           and isinstance(B, np.ndarray) and B.size==num_buses \
           and isinstance(gen_costs, np.ndarray) and gen_costs.size==num_buses:
            g_costs_only = gen_costs.flatten()[gen_indices]
            gen_deviation_vector = np.abs(B_optimized.flatten()[gen_indices] - B.flatten()[gen_indices])
            final_dev_g = np.sum(gen_deviation_vector)
            final_g_cost = np.sum(g_costs_only * gen_deviation_vector)
        elif len(gen_indices) > 0:
            print("Warning: Could not calculate final generator metrics due to invalid data/indices for summary.")
        if len(load_indices) > 0 and isinstance(B_optimized, np.ndarray) and B_optimized.size==num_buses \
           and isinstance(B, np.ndarray) and B.size==num_buses:
            load_shed_amount = np.maximum(0, B_optimized.flatten()[load_indices] - B.flatten()[load_indices])
            total_ls = np.sum(load_shed_amount)
        elif len(load_indices) > 0:
            print("Warning: Could not calculate final load shed due to invalid data/indices for summary.")
        print(f"\nInitial B (Overall):\n{B.flatten() if isinstance(B, np.ndarray) else 'N/A'}")
        print(f"\nOptimized B (Overall):\n{B_optimized.flatten() if isinstance(B_optimized, np.ndarray) else 'N/A'}")
        if isinstance(B_optimized, np.ndarray): print(f"Sum of Optimized B (should be near 0): {np.sum(B_optimized):.6f}")
        print(f"\nInitial Flows (C_unoptimized):\n{C_unoptimized.flatten() if isinstance(C_unoptimized, np.ndarray) else 'N/A'}")
        print(f"\nOptimized Flows (C_optimized):\n{C_optimized.flatten() if isinstance(C_optimized, np.ndarray) else 'N/A'}")
        print("-" * 70)
        print("\nObjective Metrics & Load Shed:")
        print(f"  Generator Deviation Sum: {final_dev_g:.4f}")
        print(f"  Generator Rescheduling Cost: {final_g_cost:.2f}")
        print(f"  Total Load Shed: {total_ls:.4f} MW (or p.u.)")
        print("\nConstraint Check Summary:")
        try:
            temp_opt = HybridPowerFlowOptimizer(A, line_limits, gen_costs, gen_limits_min, gen_limits_max, B)
            is_final_feasible = temp_opt._is_feasible(B_optimized)
            print(f"  Final solution feasible (re-checked by visualizer): {'YES' if is_final_feasible else 'NO'}")
        except Exception as e:
            print(f"Error during final feasibility re-check in visualizer: {e}")
            print("  Final solution feasibility: Unknown (re-check failed)")
    except Exception as e:
        print(f"Error generating summary text: {e}")
    finally:
        np.set_printoptions(precision=8, suppress=False)
    print("=" * 70)
    print("Note: Visualization assumes MW or consistent p.u. units. Voltage constraints are not included in this model.")
    print("=" * 70)

In [5]:
# Cell 5: Main Execution Block (Modified)

if 'HybridPowerFlowOptimizer' not in globals():
    print("ERROR: Cell 2 (HybridPowerFlowOptimizer class) not executed. Please run it first.")
    # Potentially exit or raise an error if critical components are missing
if 'optimize_power_flow_free_loadshed' not in globals():
    print("ERROR: Cell 3 (optimize_power_flow_free_loadshed function) not executed. Please run it first.")
if 'visualize_results' not in globals():
    print("ERROR: Cell 4 (visualize_results function) not executed. Please run it first.")

# This block will run when the script is executed directly.
if __name__ == "__main__":

    # --- Configuration: System Matrix File ---
    matrix_file_name = 'lock.csv'  # Name of the CSV file containing the system matrix A (e.g., PTDF)

    # --- Load System Matrix A ---
    try:
        print(f"Loading system matrix A from: {matrix_file_name}")
        # Read the CSV file using pandas, assuming no header row
        df = pd.read_csv(matrix_file_name, header=None)
        A = df.to_numpy() # Convert pandas DataFrame to NumPy array
        print(f"Successfully loaded matrix A with shape {A.shape}")
        # Validate the loaded matrix A
        if A.ndim != 2 or A.shape[0] == 0 or A.shape[1] == 0:
            raise ValueError("Loaded matrix A is not a valid 2D array or is empty.")
    except FileNotFoundError:
        print(f"FATAL ERROR: The file '{matrix_file_name}' was not found. Cannot run scenario. Exiting.")
        sys.exit() # Exit the script if the matrix file is not found
    except Exception as e: # Catch other potential errors during file loading/parsing
        print(f"FATAL ERROR occurred while reading the matrix file '{matrix_file_name}': {e}. Exiting.")
        sys.exit() # Exit on other critical errors

    # --- System Dimensions ---
    num_lines_main = A.shape[0] # Number of lines derived from matrix A
    num_buses_main = A.shape[1] # Number of buses derived from matrix A
    print(f"System dimensions: {num_lines_main} lines, {num_buses_main} buses.")

    # --- Main Scenario Loop ---
    # This loop allows the user to run multiple optimization scenarios with different inputs.
    while True:
        print("\n" + "="*25 + " New Scenario (Enhanced Hybrid) " + "="*25)
        print("Objective: 1. Meet Limits & Minimize Generator Deviation, 2. Minimize Generator Rescheduling Cost")
        print("(Load Shedding Allowed - Zero Cost Implication in Fitness for Shedding Itself)")

        try:
            # --- User Input: Line Limits ---
            print(f"\n>>> Enter LINE power limits for {num_lines_main} lines (e.g., space-separated values):")
            while True: # Loop until valid input is received
                try:
                    limits_str = input("  Line limits: ")
                    line_limits_list = [float(x) for x in limits_str.split()]
                    if len(line_limits_list) == num_lines_main:
                        line_limits_input = np.abs(np.array(line_limits_list)) # Use absolute values for limits
                        break # Valid input, exit loop
                    else:
                        print(f"  Error: Expected {num_lines_main} values, but got {len(line_limits_list)}. Please re-enter.")
                except ValueError:
                    print("  Error: Invalid number format. Please enter space-separated numbers.")
                except EOFError: raise # Propagate EOFError to exit scenario input

            # --- User Input: Initial Bus Injections (B) ---
            print(f"\n>>> Enter INITIAL power injections (B) for ALL {num_buses_main} buses:")
            print("    (Positive for Generators, Negative for Loads, e.g., space-separated)")
            while True: # Loop for valid B input
                try:
                    b_str = input("  Initial injections (B): ")
                    B_input_list = [float(x) for x in b_str.split()]
                    if len(B_input_list) == num_buses_main:
                        B_input = np.array(B_input_list).reshape(-1, 1) # Reshape to column vector
                        break # Valid input
                    else:
                        print(f"  Error: Expected {num_buses_main} values, got {len(B_input_list)}. Please re-enter.")
                except ValueError:
                    print("  Error: Invalid number format. Please enter space-separated numbers.")
                except EOFError: raise

            # Identify generator and load buses based on the user's B_input
            # This is a preliminary identification for guiding subsequent inputs. The optimizer does its own identification.
            temp_gen_indices = np.where(B_input.flatten() > 1e-6)[0] # Generators have positive injection
            temp_load_indices = np.where(B_input.flatten() <= 1e-6)[0] # Loads have non-positive injection
            if len(temp_gen_indices) == 0:
                print("\nERROR: No Generators (positive injections > 1e-6) identified in your initial B input! Cannot proceed with this scenario.")
                continue # Skip to the next iteration of the scenario loop

            print(f"-> Based on your input, identified - Generator buses: {temp_gen_indices+1}, Load buses: {temp_load_indices+1}")

            # --- User Input: Generator-Specific Data ---
            # Initialize arrays for all buses, then fill in data only for identified generator buses.
            gen_costs_input = np.zeros(num_buses_main)
            gen_limits_min_input = np.zeros(num_buses_main)
            gen_limits_max_input = np.zeros(num_buses_main)
            print(f"\n>>> Enter GENERATOR specific data ONLY for the identified generator buses: {temp_gen_indices+1}:")

            print("  Enter Generator Cost Coefficients (e.g., $/MWh of deviation):")
            for i in temp_gen_indices: # Loop through only generator buses
                while True: # Loop for valid cost input for this generator
                    try:
                        cost_str = input(f"    G{i+1} cost: ")
                        gen_costs_input[i] = float(cost_str)
                        break
                    except ValueError: print("    Invalid number. Please re-enter.")
                    except EOFError: raise

            print("  Enter Generator MINIMUM Output Limit (MW or p.u.):")
            for i in temp_gen_indices:
                while True: # Loop for valid min limit
                    try:
                        min_str = input(f"    G{i+1} min limit: ")
                        gen_limits_min_input[i] = float(min_str)
                        break
                    except ValueError: print("    Invalid number. Please re-enter.")
                    except EOFError: raise

            print("  Enter Generator MAXIMUM Output Limit (MW or p.u.):")
            for i in temp_gen_indices:
                while True: # Loop for valid max limit
                    try:
                        max_str = input(f"    G{i+1} max limit: ")
                        gen_limits_max_input[i] = float(max_str)
                        # Validate that max limit is not less than min limit for this generator
                        if gen_limits_max_input[i] < gen_limits_min_input[i]:
                            print(f"    Error: Max limit ({gen_limits_max_input[i]}) for G{i+1} cannot be less than its Min limit ({gen_limits_min_input[i]}). Please re-enter Max limit.")
                            continue # Ask for max limit again
                        break
                    except ValueError: print("    Invalid number. Please re-enter.")
                    except EOFError: raise

            # Final check for any generator having max < min after all inputs
            if np.any(gen_limits_max_input[temp_gen_indices] < gen_limits_min_input[temp_gen_indices]):
                print("\nERROR: A Generator Max limit is less than its Min limit. Please restart scenario with correct inputs.")
                continue # Restart scenario input

        except EOFError: # If user signals end-of-file (e.g., Ctrl+D) during input
            print("\nInput interrupted during scenario setup. Restarting scenario input...")
            continue
        except Exception as e: # Catch any other unexpected errors during input phase
            print(f"\nAn unexpected error occurred during input: {e}. Restarting scenario input...")
            continue

        # --- Run Optimization ---
        print("\n" + "="*25 + " Running Optimization (Enhanced Hybrid) " + "="*25)
        try:
            # Define optimization parameters
            opt_iterations = 5000  # Number of iterations for the optimizer
            opt_pop_size = 100     # Population size for the optimizer

            start_time = time.time() # Record start time
            # Call the main optimization wrapper function
            B_optimized, fitness_history, C_optimized, C_unoptimized, final_feasible, opt_details = optimize_power_flow_free_loadshed(
                A, B_input, line_limits_input, gen_costs_input, gen_limits_min_input, gen_limits_max_input,
                iterations=opt_iterations,
                population_size=opt_pop_size
            )
            end_time = time.time() # Record end time
            print(f"Optimization Duration: {end_time - start_time:.2f} seconds")

        except NameError as e: # If optimizer functions are not defined (cells not run)
            print(f"FATAL ERROR: Required function/class not defined ({e}). Make sure Cells 1-4 were executed properly.")
            break # Exit the main scenario loop
        except Exception as e: # Catch other errors during the optimization run itself
            print(f"An unexpected error occurred during the optimization run: {e}")
            try: # Ask user if they want to try another scenario despite the error
                if input("An error occurred during optimization. Try another scenario anyway? (y/n):").strip().lower() != 'y':
                    break # Exit main loop
                else:
                    continue # Start new scenario
            except EOFError:
                print("\nInput interrupted. Exiting...")
                break

        print("\n" + "="*25 + " Optimization Finished " + "="*25)

        # --- Visualize Results ---
        try:
            # Extract generator and load indices from optimizer details for visualization
            viz_gen_indices = opt_details.get("gen_indices", np.array([])) # Default to empty if not found
            viz_load_indices = opt_details.get("load_indices", np.array([]))
            print("\nVisualizing results...")
            visualize_results(A, B_input, B_optimized, C_optimized, C_unoptimized, line_limits_input,
                              gen_costs_input, gen_limits_min_input, gen_limits_max_input,
                              viz_gen_indices, viz_load_indices,
                              fitness_history)
        except NameError as e: # If visualize_results is not defined
            print(f"Error during visualization: Required function not defined ({e}). Make sure Cell 4 was executed.")
        except Exception as e: # Other visualization errors
            print(f"An error occurred during visualization: {e}")

        # --- Save Results to File (Optional) ---
        if final_feasible: # Only offer to save if the solution is feasible
            try:
                save = input("\nSave detailed results to a text file? (y/n): ").strip().lower()
                if save == 'y':
                    default_fname = "power_flow_enhanced_hybrid_results.txt"
                    fname = input(f"Enter filename (default: {default_fname}): ").strip()
                    if not fname: fname = default_fname # Use default if user enters nothing

                    print(f"Attempting to save results to {fname}...")
                    try:
                        with open(fname, 'w') as f:
                            # Calculate metrics again for saving, ensuring correct indices are used
                            gen_dev_save = 0.0; gen_cost_save_val = 0.0; load_shed_save = 0.0
                            gen_costs_for_saving = []

                            if len(viz_gen_indices) > 0:
                                # Ensure indices are valid before accessing arrays
                                if np.all(viz_gen_indices < num_buses_main):
                                    gen_costs_for_saving = gen_costs_input[viz_gen_indices]
                                    dev_vec_save = np.abs(B_optimized.flatten()[viz_gen_indices] - B_input.flatten()[viz_gen_indices])
                                    gen_dev_save = np.sum(dev_vec_save)
                                    if len(gen_costs_for_saving) == len(dev_vec_save):
                                         gen_cost_save_val = np.sum(gen_costs_for_saving * dev_vec_save)
                                    else: print("Warning (save): Mismatch in lengths for gen cost calculation.")
                                else: print("Warning (save): Invalid generator indices for metric calculation.")


                            if len(viz_load_indices) > 0 :
                                if np.all(viz_load_indices < num_buses_main):
                                    shed_vec_save = np.maximum(0, B_optimized.flatten()[viz_load_indices] - B_input.flatten()[viz_load_indices])
                                    load_shed_save = np.sum(shed_vec_save)
                                else: print("Warning (save): Invalid load indices for metric calculation.")


                            f.write("Power Flow Optimization Results (Enhanced Hybrid - Load Shed Allowed - Zero Cost)\n")
                            f.write("Objective: 1. Meet Limits & Minimize Generator Deviation, 2. Minimize Rescheduling Cost\n")
                            f.write("="*30+"\n\n")
                            f.write(f"System Matrix A (Shape: {A.shape}):\n"); np.savetxt(f, A, fmt='%.4f'); f.write("\n") # Save matrix A
                            f.write("Line Limits:\n"); [f.write(f"L{i+1}: {l:.2f}\n") for i,l in enumerate(line_limits_input)]
                            f.write("\nGenerator Costs (for deviation):\n")
                            if len(viz_gen_indices) == len(gen_costs_for_saving): # Check if gen_costs_for_saving was populated correctly
                                [f.write(f"G{viz_gen_indices[i]+1}: {c:.2f}\n") for i,c in enumerate(gen_costs_for_saving)]
                            else: f.write("  Could not save generator costs due to index issues.\n")

                            f.write("\nGenerator Min Limits:\n"); [f.write(f"G{idx+1}: {gen_limits_min_input[idx]:.2f}\n") for idx in viz_gen_indices if idx < num_buses_main]
                            f.write("\nGenerator Max Limits:\n"); [f.write(f"G{idx+1}: {gen_limits_max_input[idx]:.2f}\n") for idx in viz_gen_indices if idx < num_buses_main]

                            f.write("\nInitial B (Bus Injections):\n"); [f.write(f"Bus {i+1}: {b[0]:.4f}\n") for i,b in enumerate(B_input)]
                            f.write("\nOptimized B (Bus Injections):\n"); [f.write(f"Bus {i+1}: {b[0]:.4f}\n") for i,b in enumerate(B_optimized)]
                            f.write("\nInitial C (Line Flows):\n"); [f.write(f"L{i+1}: {c[0]:.4f}\n") for i,c in enumerate(C_unoptimized)]
                            f.write("\nOptimized C (Line Flows):\n"); [f.write(f"L{i+1}: {c[0]:.4f}\n") for i,c in enumerate(C_optimized)]
                            f.write(f"\nFinal Generator Deviation Sum: {gen_dev_save:.4f}\n")
                            f.write(f"Final Generator Rescheduling Cost: {gen_cost_save_val:.2f}\n")
                            f.write(f"Total Load Shed: {load_shed_save:.4f} MW (or p.u.)\n")
                            f.write(f"\nFinal Solution Feasible: {'YES' if final_feasible else 'NO'}\n")
                            print(f"Results successfully saved to {fname}")
                    except IOError as e:
                        print(f"ERROR saving results to file '{fname}': {e}")
                    except IndexError as e: # Catch potential errors if indices are bad during saving
                        print(f"ERROR saving results: Index out of bounds - {e}. Check consistency of generator/load indices.")
                    except Exception as e: # Catch any other saving error
                        print(f"An unexpected error occurred during saving results: {e}")
            except EOFError:
                print("\nInput interrupted during save prompt.")
                # Ask if user wants to continue to next scenario even if save was interrupted
                if input("Continue to next scenario anyway? (y/n):").strip().lower() != 'y':
                    break # Exit main loop
        elif not final_feasible:
            print("\nFinal solution was infeasible. Results not saved.")

        # --- Ask to Run Another Scenario ---
        try:
            run_again = input("\nRun another scenario? (y/n):").strip().lower()
            if run_again != 'y':
                print("\nExiting script...")
                break # Exit the main `while True` scenario loop
            else:
                print("\nRestarting for new scenario...\n" + "-"*70)
        except EOFError: # Handle Ctrl+D during this prompt
            print("\nInput interrupted. Exiting script...")
            break # Exit main loop

    # --- Script End ---
    print("\nScript finished.")


ERROR: Cell 4 (visualize_results function) not executed. Please run it first.
Loading system matrix A from: lock.csv
Successfully loaded matrix A with shape (20, 14)
System dimensions: 20 lines, 14 buses.

========================= New Scenario (Enhanced Hybrid) =========================
Objective: 1. Meet Limits & Minimize Generator Deviation, 2. Minimize Generator Rescheduling Cost
(Load Shedding Allowed - Zero Cost Implication in Fitness for Shedding Itself)

>>> Enter LINE power limits for 20 lines (e.g., space-separated values):


KeyboardInterrupt: Interrupted by user

In [6]:
219 18.3 -94.2 -47.8 -7.6 -11.2 0 0 -29.5 -9 -3.5 -6.1 -13.5 -14.9 # case 1

SyntaxError: invalid syntax (1888858573.py, line 1)

In [ ]:
170 100 125 100 50 100 100 100 100 100 100 100 15 100 100 100 100 100 100 100 # limits

In [ ]:
241 28.3 -94.2 -47.8 -47.6 -11.2 0 0 -29.50 -9 -3.5 -6.1 -5.5 -14.9 # case 2

In [ ]:
235 61.3 -214.2 -1.4 -1 -11.2 0 0 -29.50 -9 -3.5 -6.1 -5.5 -14.9 # case 3